[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C33_Context_Memory_Course/02_compaction/02_compaction.ipynb)

# 02 · Compaction 与摘要

目标：用**纯标准库**从零写出对话 compaction 的完整 scaffold——**触发阈值 → 切分(摘要远期/保留近期) → 关键信息保留 → 压缩比 → 滚动摘要**，全程用 **MockLLM** 当模型、`assert` 验证，**无需 API key**。

路线：复用计数与 MockLLM → 触发判定 → compaction 器(摘要旧+保近期) → 关键 id 保留 → 滚动摘要 → ✏️ 练习 → 📖 答案 → 🧪 真实长会话胶囊 → 真实 compaction 形状对照。

> 心智模型：**compaction = 历史快塞满时，把远期旧消息打包摘要成一块、近期原文原样留下，于是历史从『线性增长』变成『有界』**。难点在阈值、切分、保留、度量四个决策。

## 1 · 复用地基：近似计数 + 会摘要的 MockLLM

先把模块 00 的两件地基搬进来，让本 notebook **自包含可独立运行**：确定性近似 `count_tokens`/`count_messages`，以及一个会 `summarize` 的 **MockLLM**。摘要必须**确定性**（要点抽取 + 拼接），这样 `assert` 才能验证。

In [ ]:
import json, re

def count_tokens(text):
    '''确定性近似: 按空白切词 + 长词拆分。真实请改用 messages.count_tokens。'''
    if not text:
        return 0
    return sum(max(1, (len(w) + 3) // 4) for w in text.split())

def count_messages(messages):
    '''一组消息(role+content)的近似 token, 含每条角色开销。'''
    total = 0
    for m in messages:
        total += 4
        cont = m['content'] if isinstance(m['content'], str) else json.dumps(m['content'], ensure_ascii=False)
        total += count_tokens(cont)
    return total

class MockLLM:
    '''确定性假模型: summarize = 抽取要点 + 直通关键 id。'''
    def __init__(self):
        self.summarize_calls = 0
    def summarize(self, text, max_points=4):
        '''① 抽取每行首句的前若干为要点; ② 把所有 FACT#\d+ 关键 id 原样直通。'''
        self.summarize_calls += 1
        lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
        pts = [ln.split('。')[0].split('. ')[0][:30] for ln in lines[:max_points]]
        ids = sorted(set(re.findall(r'FACT#\d+', text)))   # 关键 id 直通, 不靠自由摘要碰运气
        out = '要点: ' + '; '.join(pts)
        if ids:
            out += ' || 关键: ' + ' '.join(ids)
        return out

llm = MockLLM()
# 一段啰嗦的多轮对话(每行远超摘要的截断长度, 且行数 > max_points), 摘要才有意义
demo = ('用户说他的预算上限是 1000 元, 希望尽量不要超过这个数字。FACT#42 这是硬约束\n'
        '助手详细比较了方案 A 与方案 B 的差异并最终确认采用方案 A。FACT#7 已定方案\n'
        '用户补充说付款后一定要开一张增值税专用发票寄到公司地址。\n'
        '助手回答说没问题会在发货时一并把发票寄出请放心等待。\n'
        '用户又问了大概多少天能到货以及是否支持七天无理由退换。\n'
        '助手说一般三到五个工作日送达并且支持七天无理由退换货。')
s = llm.summarize(demo)
print('摘要:', s)
print('摘要前 token:', count_tokens(demo), '| 摘要后 token:', count_tokens(s))
assert 'FACT#42' in s and 'FACT#7' in s        # 关键 id 必须直通
assert count_tokens(s) < count_tokens(demo)    # 摘要确实更短(啰嗦原文被压成要点)
assert llm.summarize_calls == 1
print('✅ 地基就位: 确定性摘要会抽要点、且把关键 id 原样保留')

## 2 · 触发判定：到水位线才压

compaction 不是每轮都做，而是**历史 token 超过『可用预算』的某比例**才触发。关键：比例算在 **窗口 − 输出预留** 这块可用预算上（和模块 01『先扣输出预留』同一条纪律），不是整个窗口。

In [ ]:
def should_compact(history, window, reserve_output, trigger=0.7):
    '''历史占可用预算(window-reserve)的比例 >= trigger 就该压。
       返回 (是否该压, 当前占比)。'''
    usable = window - reserve_output
    used = count_messages(history)
    ratio = used / usable
    return ratio >= trigger, ratio

# 造一段历史: 1 条系统 + 若干轮对话
def make_history(n_turns, words_each=20):
    h = [{'role': 'system', 'content': 'you are a helpful agent', 'pinned': True}]
    for i in range(n_turns):
        h.append({'role': 'user', 'content': ('word ' * words_each).strip()})
        h.append({'role': 'assistant', 'content': ('reply ' * words_each).strip()})
    return h

short = make_history(3)     # 还很空
longh = make_history(40)    # 涨满了
do_s, r_s = should_compact(short, window=2000, reserve_output=500)
do_l, r_l = should_compact(longh, window=2000, reserve_output=500)
print(f'短历史 占比={r_s:.2f} -> 压? {do_s}')
print(f'长历史 占比={r_l:.2f} -> 压? {do_l}')
assert do_s is False and do_l is True       # 短的不压, 长的要压
assert r_l > r_s                              # 占比单调
# 阈值越低越容易触发
assert should_compact(short, 2000, 500, trigger=0.05)[0] is True
print('✅ 触发判定: 按『历史/可用预算』比例, 到阈值才压, 且对输出预留留了余地')

## 3 · compaction 器：摘要远期、保留近期

触发后：**钉住项**(系统提示)永不摘要；**最近 `keep_recent` 条**原文保留；**其余旧消息**合并交给 `llm.summarize`，结果作为一条 `[压缩记忆]` 消息插在『钉住 + 摘要 + 近期』之间。

In [ ]:
def compact(messages, llm, keep_recent=4):
    '''返回压缩后的消息序列: pinned + [摘要块] + 近期原文。
       旧消息不足以压(去掉钉住+近期后<=0)时, 原样返回。'''
    pinned = [m for m in messages if m.get('pinned')]
    rest   = [m for m in messages if not m.get('pinned')]
    if len(rest) <= keep_recent:
        return list(messages)                       # 不值得压
    old, recent = rest[:-keep_recent], rest[-keep_recent:]
    blob = '\n'.join(m['content'] for m in old)
    summary_msg = {'role': 'user', 'content': '[压缩记忆] ' + llm.summarize(blob)}
    return pinned + [summary_msg] + recent

hist = make_history(20)                              # 1 系统 + 40 条对话
before = count_messages(hist)
comp = compact(hist, llm, keep_recent=4)
after = count_messages(comp)
print(f'压缩前 {len(hist)} 条 / {before} tok  ->  压缩后 {len(comp)} 条 / {after} tok')
# 不变量: 钉住项还在 + 近期 4 条原样在 + 总 token 下降 + 有一个摘要块
assert comp[0].get('pinned') is True                 # 系统提示保住
assert comp[-4:] == hist[-4:]                        # 最近 4 条原文不动
assert any('[压缩记忆]' in m['content'] for m in comp)
assert after < before                                # 确实腾出了空间
assert len(comp) < len(hist)                         # 条数减少
# 边界: 旧消息不够多时不压
assert compact(make_history(1), llm, keep_recent=4) == make_history(1)
print('✅ compaction 器: 钉住保住、近期原样、远期压成一块、空间下降')

## 4 · 端到端: 边对话边在触发时压缩

把触发判定 + compaction 器接成一个**会自我维护的对话循环**: 每加一轮就检查是否该压, 该压就压。这正是长会话 agent 的上下文管理主回路——历史因此**有界**, 永不爆窗。

In [ ]:
def chat_turn(history, user_msg, llm, window, reserve_output, trigger=0.7, keep_recent=4):
    '''加一轮用户消息(+一个占位回复), 然后按需 compaction。返回新历史。'''
    history = history + [
        {'role': 'user', 'content': user_msg},
        {'role': 'assistant', 'content': 'ok, ' + ('detail ' * 15).strip()},
    ]
    do, _ = should_compact(history, window, reserve_output, trigger)
    if do:
        history = compact(history, llm, keep_recent)
    return history

h = [{'role': 'system', 'content': 'agent', 'pinned': True}]
peak = 0
for i in range(60):                                  # 聊 60 轮
    h = chat_turn(h, f'question {i} ' + ('x ' * 18).strip(), llm,
                  window=1500, reserve_output=400, trigger=0.7, keep_recent=4)
    peak = max(peak, count_messages(h))
usable = 1500 - 400
print(f'聊了 60 轮, 历史峰值 token={peak}, 可用预算={usable}')
print(f'最终历史 {len(h)} 条 (而非 121 条) —— 被 compaction 控制住了')
assert peak <= usable                                # 关键: 全程没爆可用预算!
assert len(h) < 121                                  # 远少于不压时的 1+60*2
assert h[0].get('pinned') is True                    # 系统提示始终在
print('✅ 端到端: 60 轮对话历史有界、never 爆窗 —— compaction 主回路成立')

## 5 · 滚动摘要: 旧摘要 + 增量 再摘要

连续触发时, 把**上一份摘要 + 这批新变旧的消息**一起再摘要成新的一份。摘要本身**有界**(不随对话变长), 但自由散文部分会**逐渐漂移**(传话游戏)。关键 id 走『直通』通道, 不进传话链, 所以始终不丢。

In [ ]:
def rolling_compact(prev_summary, new_old_messages, llm):
    '''prev_summary: 上一份摘要文本(可为''); new_old_messages: 本轮变旧的消息。
       返回新摘要文本。'''
    blob = prev_summary + '\n' + '\n'.join(m['content'] for m in new_old_messages)
    return llm.summarize(blob)

# 模拟 5 轮滚动: 第1轮埋入关键 id, 看它能否一路活到最后
summary = ''
batches = [
    [{'content': '用户定预算上限 1000。FACT#42'}, {'content': '确认方案 A。'}],
    [{'content': '用户要发票。FACT#7'}, {'content': '已记录。'}],
    [{'content': '讨论配色。'}, {'content': '选深色。'}],
    [{'content': '讨论排期。'}, {'content': '两周内。'}],
    [{'content': '用户确认下单。'}, {'content': '完成。'}],
]
lengths = []
for b in batches:
    summary = rolling_compact(summary, b, llm)
    lengths.append(count_tokens(summary))
print('每轮摘要 token:', lengths)
print('最终摘要:', summary)
# 有界: 摘要不随轮数爆炸增长 (最后一轮不比最大的多很多)
assert max(lengths) <= lengths[0] * 4                # 大致有界, 非线性累加
# 关键 id 一路存活 (走直通, 不被传话游戏吃掉)
assert 'FACT#42' in summary and 'FACT#7' in summary
print('✅ 滚动摘要: 摘要有界 + 关键 id 全程存活(直通通道躲过漂移)')

---
## ✏️ 练习 1：摘要触发——压到目标水位以下

第 2 节的 `should_compact` 只判断『要不要压』。实战还要避免**抖动**：压完应当降到一个**目标水位**以下, 否则压完还在阈值边缘、下一轮立刻又触发。

实现 `compact_until(history, llm, window, reserve_output, trigger, target, keep_recent)`：
若占比 `>= trigger` 就反复 `compact` **直到** 占比 `<= target` **或** 已压不动(条数不再减少)；返回 `(新历史, 压了几次)`。

In [ ]:
def compact_until(history, llm, window, reserve_output,
                  trigger=0.7, target=0.5, keep_recent=4):
    # TODO:
    #  usable = window - reserve_output
    #  rounds = 0
    #  若 count_messages(history)/usable < trigger: 直接返回 (history, 0)
    #  循环: 记录压前条数; history = compact(...); rounds += 1
    #        若 count_messages/usable <= target: break
    #        若 条数没再减少(压不动了): break
    #  返回 (history, rounds)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
h = make_history(50)                                  # 很满
usable = 1500 - 400
before_ratio = count_messages(h) / usable
newh, rounds = compact_until(h, llm, 1500, 400, trigger=0.7, target=0.5, keep_recent=4)
after_ratio = count_messages(newh) / usable
print(f'压前占比={before_ratio:.2f} -> 压后占比={after_ratio:.2f}, 压了 {rounds} 次')
assert before_ratio >= 0.7                            # 前提: 确实超阈值
assert rounds >= 1
assert after_ratio <= 0.5 or rounds >= 1              # 压到目标以下(或已尽力)
# 不触发时不压
assert compact_until(make_history(2), llm, 1500, 400)[1] == 0
print('✅ 练习 1 通过: 压到目标水位以下、避免抖动、不触发不压')

## ✏️ 练习 2：滚动摘要——摘要长度有界

证明滚动摘要『有界』: 连续摘要很多轮, 摘要的 token 数**不随轮数线性增长**。

实现 `roll_n(batches, llm)`：对一串 `batches`(每个是若干消息)依次做 `rolling_compact`, 返回**每一轮之后摘要的 token 数组成的列表**。

In [ ]:
def roll_n(batches, llm):
    # TODO: summary='' ; 对每个 batch 调 rolling_compact 更新 summary;
    #       收集 count_tokens(summary) 到列表并返回
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
many = [[{'content': f'第{i}轮讨论某事。'}, {'content': f'回复{i}。'}] for i in range(12)]
lens = roll_n(many, llm)
print('12 轮摘要 token 序列:', lens)
assert len(lens) == 12
# 有界: 第 12 轮不比第 1 轮的几倍更多 (而非线性累加到 12 倍)
assert lens[-1] <= lens[0] * 5
# 对比: 不摘要直接拼接会线性爆炸
naive = count_tokens('\n'.join(m['content'] for b in many for m in b))
assert lens[-1] < naive                               # 摘要确实比全量拼接短
print('✅ 练习 2 通过: 滚动摘要有界, 远小于全量拼接')

## ✏️ 练习 3：关键信息保留——压缩后关键 id 不丢

compaction 最重要的不变量: **该记住的关键 id 压完还在**。

实现 `key_ids_survive(messages, llm, keep_recent)`：先用正则从**原始** `messages` 抽出所有 `FACT#\d+` 作为关键集合 `K`；`compact` 之后, 从压缩结果的**全部可见文本**里抽出 id 集合 `V`；返回 `(K 是否都在 V 里, 丢失的 id 集合)`。

In [ ]:
def key_ids_survive(messages, llm, keep_recent=4):
    # TODO:
    #  K = set(re.findall(r'FACT#\d+', 把所有原始消息 content 拼起来))
    #  comp = compact(messages, llm, keep_recent)
    #  V = set(re.findall(r'FACT#\d+', 把 comp 全部 content 拼起来))
    #  返回 (K <= V, K - V)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
msgs = [{'role': 'system', 'content': 'agent', 'pinned': True}]
msgs += [{'role': 'user', 'content': f'要点{i} FACT#{i}'} for i in range(10)]   # 埋 10 个 id 在旧消息
msgs += [{'role': 'user', 'content': '最近一句, 无 id'}]
ok, lost = key_ids_survive(msgs, llm, keep_recent=2)
print('全部关键 id 存活?', ok, '| 丢失:', lost)
assert ok is True and lost == set()                  # 一个都不能丢
# 反例: 如果摘要器不直通 id(假装自由摘要丢了它们), 就该报出丢失
class LossyLLM:
    def summarize(self, text, max_points=4):
        return '要点(自由摘要, 丢了 id)'
ok2, lost2 = key_ids_survive(msgs, LossyLLM(), keep_recent=2)
assert ok2 is False and len(lost2) > 0                # 检测到沉默丢失!
print('✅ 练习 3 通过: 关键 id 不变量可验证, 且能抓出沉默丢失')

## ✏️ 练习 4：压缩比——压完省了多少

度量一次 compaction 的强度。

实现 `compression_ratio(old_messages, summary_text)`：返回 `count_tokens(summary_text) / count_messages(old_messages)`(被摘要那部分用 `count_messages`, 摘要是纯文本用 `count_tokens`)。若被摘要部分为空(0 token)返回 `1.0`(没东西可压)。

In [ ]:
def compression_ratio(old_messages, summary_text):
    # TODO: denom = count_messages(old_messages); 若 denom==0 返回 1.0;
    #       否则返回 count_tokens(summary_text)/denom
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
old = [{'role': 'user', 'content': ('detail ' * 30).strip()} for _ in range(8)]
summ = llm.summarize('\n'.join(m['content'] for m in old))
ratio = compression_ratio(old, summ)
print(f'被摘要 {count_messages(old)} tok -> 摘要 {count_tokens(summ)} tok, 压缩比={ratio:.3f}')
assert 0.0 < ratio < 1.0                              # 确实压小了
assert compression_ratio([], 'anything') == 1.0       # 空: 无可压
# 压缩比越小=压得越狠: 更长的旧历史压成同量级摘要 -> 比值更小
longer = [{'role': 'user', 'content': ('detail ' * 30).strip()} for _ in range(40)]
assert compression_ratio(longer, summ) < ratio
print('✅ 练习 4 通过: 压缩比可算、落在 (0,1)、且越长压得越狠')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def compact_until(history, llm, window, reserve_output,
                  trigger=0.7, target=0.5, keep_recent=4):
    usable = window - reserve_output
    if count_messages(history) / usable < trigger:
        return history, 0
    rounds = 0
    while True:
        prev_len = len(history)
        history = compact(history, llm, keep_recent)
        rounds += 1
        if count_messages(history) / usable <= target:
            break
        if len(history) >= prev_len:        # 压不动了
            break
    return history, rounds

In [ ]:
# 练习 2 参考答案
def roll_n(batches, llm):
    summary, out = '', []
    for b in batches:
        summary = rolling_compact(summary, b, llm)
        out.append(count_tokens(summary))
    return out

In [ ]:
# 练习 3 参考答案
def key_ids_survive(messages, llm, keep_recent=4):
    K = set(re.findall(r'FACT#\d+', ' '.join(m['content'] for m in messages)))
    comp = compact(messages, llm, keep_recent)
    V = set(re.findall(r'FACT#\d+', ' '.join(m['content'] for m in comp)))
    return K <= V, K - V

In [ ]:
# 练习 4 参考答案
def compression_ratio(old_messages, summary_text):
    denom = count_messages(old_messages)
    if denom == 0:
        return 1.0
    return count_tokens(summary_text) / denom

---
## 🧪 真实数据胶囊：一段真实量级的长会话压缩

下面是一段**贴近真实**的客服长会话(含用户硬约束 + 关键订单号), 量级接近真实场景。我们用本课的 compaction 器把它压一遍, 验证: **关键约束/订单号不丢、压缩比合理、压后塞得进一个小窗口**。

> 形状对照: 真实里这些就是 Messages API 的 `messages` 数组; `summarize` 那一步就是 `client.messages.create(model='claude-sonnet-4-6', ...)` 让模型写摘要。

In [ ]:
# 真实风格的长会话(节选 + 程序化扩展到真实量级)
REAL_CHAT = [
    {'role': 'system', 'content': '你是某电商的客服 agent, 严格遵守用户约束。', 'pinned': True},
    {'role': 'user', 'content': '我的预算上限是 800 元, 不要超。订单号 FACT#100861。'},
    {'role': 'assistant', 'content': '好的, 已记录预算 800 与订单 FACT#100861。'},
]
# 程序化加入大量中间闲聊(模拟几十轮), 把历史撑大
for i in range(30):
    REAL_CHAT.append({'role': 'user', 'content': f'顺便问一下第{i}个小问题, ' + ('闲聊 ' * 12).strip()})
    REAL_CHAT.append({'role': 'assistant', 'content': f'关于第{i}点, ' + ('解释 ' * 12).strip()})
# 最近一轮: 用户要下单
REAL_CHAT.append({'role': 'user', 'content': '现在帮我按之前的约束下单。'})

before = count_messages(REAL_CHAT)
real_llm = MockLLM()
comp = compact(REAL_CHAT, real_llm, keep_recent=3)
after = count_messages(comp)
vis = ' '.join(m['content'] for m in comp)
print(f'真实会话 {len(REAL_CHAT)} 条 / {before} tok -> 压后 {len(comp)} 条 / {after} tok')
print('压后能塞进 600-token 窗口?', after <= 600)
assert 'FACT#100861' in vis            # 关键订单号没丢
assert '800' in vis                     # 关键预算约束没丢
assert after < before                   # 确实压小
assert comp[0].get('pinned') is True    # 系统约束保住
print('✅ 胶囊: 真实量级长会话压缩后, 关键约束/订单号保住、塞得进小窗口')

**🧪 胶囊练习**：实现 `health_check(old_messages, summary_text, lo=0.1, hi=0.6)`：算出压缩比, 返回 `'ok'`(在 [lo,hi])/`'too_weak'`(>hi, 几乎没压)/`'too_aggressive'`(<lo, 压过头)。(真实系统就是这样给自己的 compaction 做体检。)

In [ ]:
def health_check(old_messages, summary_text, lo=0.1, hi=0.6):
    # TODO: r = compression_ratio(old_messages, summary_text)
    #       r > hi -> 'too_weak'; r < lo -> 'too_aggressive'; 否则 'ok'
    raise NotImplementedError

In [ ]:
# 自测
old = [{'role': 'user', 'content': ('detail ' * 30).strip()} for _ in range(20)]
good = real_llm.summarize('\n'.join(m['content'] for m in old))
assert health_check(old, good) in ('ok', 'too_aggressive')   # 压得明显
assert health_check(old, ' '.join(m['content'] for m in old)) == 'too_weak'  # 原样=没压
print('压缩比健康度:', health_check(old, good))
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def health_check(old_messages, summary_text, lo=0.1, hi=0.6):
    r = compression_ratio(old_messages, summary_text)
    if r > hi:
        return 'too_weak'
    if r < lo:
        return 'too_aggressive'
    return 'ok'

---
## 🔧 旁注：真实的 compaction 长什么样（含一个致命坑）

本课手写的 compaction 器, 对应到真实 Anthropic API 有两条路:

**① 自己摘要(手动 compaction)** —— 把 `llm.summarize(旧消息)` 换成一次真实模型调用让它写摘要(伪代码, **本环境不跑、需 API key**):

```python
import anthropic
client = anthropic.Anthropic()                 # 读 ANTHROPIC_API_KEY
summary = client.messages.create(
    model='claude-sonnet-4-6', max_tokens=512,
    system='把以下对话压成要点, 务必保留用户约束/决策/订单号等关键信息。',
    messages=[{'role':'user','content': blob_of_old_messages}],
).content[0].text
# 然后: new_messages = pinned + [{'role':'user','content':'[压缩记忆] '+summary}] + recent
```

**② 服务端 compaction(beta `compact-2026-01-12`)** —— API 接近窗口上限时**自动**压缩, 但有一个**致命坑**:

```python
resp = client.beta.messages.create(
    betas=['compact-2026-01-12'], model='claude-sonnet-4-6', max_tokens=1024,
    messages=messages,
    context_management={'edits': [{'type': 'compact_20260112'}]},
)
messages.append({'role':'assistant', 'content': resp.content})  # ← 必须 append 整个 content!
```

> ‼️ **铁律**: 服务端 compaction 会在 `resp.content` 里放一个 **compaction 块**, 你必须把**整个 `resp.content`** append 回 `messages`(而不是只取 `.content[0].text`)。只取文本会**悄悄丢掉压缩状态**, 下一轮 API 无法接续压缩——这是最常见、最难查的坑。本课模块 02 的手写器即其最小内核。

### 小结
- compaction = 历史快塞满时, **摘要远期旧消息 + 保留近期原文**, 让历史从线性增长变**有界**。
- **触发**: 按『历史 / 可用预算(窗口−输出预留)』比例到阈值才压; 压到目标水位以下避免抖动。
- **切分**: 钉住项(系统提示)永不摘要; 最近 `keep_recent` 条原样保留; 其余压成一块。
- **关键信息保留**是最重要的不变量: 用『关键 id 压后仍可检出』做成 `assert`, 抓出**沉默丢失**。
- **滚动摘要**有界但会**漂移**(传话游戏); 关键 id 走直通通道躲过漂移。
- **压缩比** = 摘要后/摘要前 token, 既是效果度量也是健康检查(太接近 1 没压动、太接近 0 压过头)。
- 真实对照: 手动摘要(`messages.create`)或服务端 compaction(`compact-2026-01-12`); 后者**务必 append 整个 `response.content`**。

下一站：**模块 03 · 文件记忆** —— 把『该长期记住但不必时刻在场』的东西移出窗口、按需召回。